# 00 — Setup and map of the PII touchpoints

**Goal of this series.** An agent runtime (a LangGraph orchestrator receiving requests over A2A and REST,
calling an LLM, sub-agents, REST APIs and MCP tools, with memory and observability) handles personal data
at several places. Each notebook takes **one touchpoint**, shows what Azure offers for it according to the
current documentation, and demonstrates it on **synthetic data only**.

**How to use.** Open the notebooks in VS Code, select the kernel of the repository environment
(`.conda/python.exe`) and run cells from top to bottom. Azure resources are created by
`scripts/deploy.sh` (see `README.md`); nothing in the notebooks creates or deletes resources.

## 1. The runtime and its touchpoints

```
                      (1) inputs                                     (2) LLM call
 user / app ──HTTP──► IN A2A / IN REST(JSON) ──► Orchestrator ──────────► LLM
                                                  (LangGraph)
                                                   │    │    │
                     (3) sub-agents (A2A), REST APIs, MCP tools ── data goes out AND comes back
                                                   │
          (5) short-term memory (Redis)  ·  long-term memory (vector index)
          (4) observability (OpenTelemetry ──► Application Insights)
          (6) knowledge base (RAG index)   ·  (7) audit and analytics
```

| # | Touchpoint | Question | Notebook |
|---|---|---|---|
| 1 | Inputs | Can we detect personal data in prompts, JSON payloads and images, including travel identifiers, in several languages, without false positives? | `01_inputs` |
| 1 | Inputs, with Presidio | Same question with the open-source Presidio framework: default settings, then configured | `01b_presidio` |
| 2 | LLM call | Can the LLM work on masked data, and can we put the real values back afterwards? | `02_llm_call` |
| 3 | Sub-agents, APIs, MCP | Does each agent or tool receive only the personal data its mission needs, in and out? | `03_agents_tools` |
| 4 | Observability | Do logs and traces stay free of personal data, before they reach Application Insights? | `04_observability` |
| 5 | Memory | What do we store in short-term and long-term memory, and where do real values live? | `05_memory` |
| 6 | Knowledge base | Can personal data be kept out of a RAG index at indexing time? | `06_knowledge_db` |
| 7 | Audit | Can we audit transactions while keeping only legitimate personal data? | `07_audit` |

## 2. Six words used in every notebook

| Word | Meaning here |
|---|---|
| **Detection** | Finding where personal data is in a text, a JSON payload or an image, and of which type. |
| **Masking** | Replacing a value with `***` or `[PERSON]`. One-way: the value cannot be recovered. |
| **Pseudonymization** | Replacing a value with a token such as `[PERSON_1]` and keeping the mapping in a protected store, so that it can be reversed. Under the GDPR (Art. 4(5)), pseudonymized data is **still personal data**. |
| **Anonymization** | Irreversible removal of anything that identifies a person. Hard to achieve; not the goal of this series. |
| **Re-identification** | Putting the real value back, only where it is needed (for example the booking tool needs the real name). |
| **Minimization** | Each component receives only the personal data it needs (GDPR Art. 5(1)(c)). |

## 3. Personal data in scope

| Type | Synthetic example | Detected by |
|---|---|---|
| Passenger name, group name, title | `Jane Whitfield`, `Blue Heron Travellers`, `Mrs` | Azure Language (`Person`); group name and title: our rules |
| Passport and national documents | `523941876` | Azure Language (passport and national ID categories) |
| Phone, e-mail, address, date of birth | `+44 20 7946 0958`, `jane.whitfield@example.com` | Azure Language |
| **Booking reference (PNR)** | `K7M2QR` (pattern `^[A-Z2-9]{6}$`) | No Azure category: our rules (notebook 01) |
| **Hotel confirmation number** | `a889Ce6FFF8d0` (pattern `[0-9a-zA-Z]{2,20}`) | No Azure category: our rules |
| **Loyalty number** | `XY 4815162342` | No Azure category: our rules |

**Labels used in the notebooks:** **[Documented]** = stated on Microsoft Learn (link given);
**[Observed]** = what the cell output shows on the day you run it; **[Assumption]** = to be validated.

## 4. Environment check

In [ ]:
import sys
from importlib.metadata import version

print("Python", sys.version.split()[0])
for package in ["azure-identity", "httpx", "pillow", "ipython"]:
    print(f"{package:15} {version(package)}")

## 5. Azure configuration and sign-in

In [ ]:
import pii_lab as L

env = L.load_env()
print("PII_FOUNDRY_ENDPOINT configured:", bool(env.get("PII_FOUNDRY_ENDPOINT")))
token = L._token()   # Entra ID token from `az login`: no API key anywhere
print("Azure CLI sign-in: OK (token acquired)")

## 6. First call: is Azure Language reachable?

One short synthetic sentence. If this cell fails with HTTP 401/403 right after deployment,
wait a few minutes: role assignments take time to propagate.

In [ ]:
doc = L.language_pii(["Please call Jane Whitfield on +44 20 7946 0958."])[0]
for e in doc["entities"]:
    print(f'{e["category"]:12} {e["text"]:20} confidence={e["confidenceScore"]}')
print("redactedText:", doc["redactedText"])

Next: **`01_inputs`** — detection on free text, JSON payloads and images.